# Statische Prognose einer Sinusschwingung mit einem neuronalen Netz

**Modul:** Cyber-Physische Systeme – Prognose und Planung

## Lernziel
Sie können ein **statisches** Prognosemodell (fully connected Netz) trainieren und es **auto-regressiv**
einsetzen, um weiter in die Zukunft zu prognostizieren als das Modell trainiert wurde.

## Aufbau
- **Eingang:** 5 Samples der Sinusschwingung, jeweils im Abstand von $30°$
  $\;u_{t-4}, u_{t-3}, u_{t-2}, u_{t-1}, u_t$ (5 Eingangsneuronen).
  Jedes Sample steht für ein $30°$-Intervall, die 5 Samples decken damit ein Fenster von $5 \cdot 30° = 150°$ ab.
- **Ausgang:** der Wert $y_{t+1}$ der Sinuskurve **30° in der Zukunft** (1 Ausgangsneuron).
- **Netz:** fully connected: $5 \rightarrow$ versteckte Schichten $\rightarrow 1$.

> Das Abtastintervall von $30°$ ist zugleich der Prognosehorizont. Nur so lässt sich die Prognose
> wieder als Eingang verwenden (siehe unten).

## Auto-regressive Prognose
Um 60° oder 90° vorherzusagen, wird die Prognose als neuer Eingang zurück ins Modell gegeben
und dabei der älteste Datenpunkt verworfen:

```
u_{t-4} u_{t-3} u_{t-2} u_{t-1} u_t     → Modell → y_{t+1}   (+30°)
u_{t-3} u_{t-2} u_{t-1} u_t     y_{t+1} → Modell → y_{t+2}   (+60°)
u_{t-2} u_{t-1} u_t     y_{t+1} y_{t+2} → Modell → y_{t+3}   (+90°)
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neural_network import MLPRegressor

## 1. Parameter  ✏️

In [ ]:
STEP_DEG = 30        # Abstand der Samples = Prognosehorizont pro Schritt in Grad
N_INPUTS = 5         # Anzahl Eingangsneuronen (Samples im Eingangsfenster)
N_TRAIN = 3000       # Anzahl Trainingsbeispiele
HIDDEN = (16, 16)    # Neuronen der versteckten Schichten
NOISE_STD = 0.0      # Rauschen auf den Trainings-Eingängen (z. B. 0.02 ausprobieren)
SEED = 42

rng = np.random.default_rng(SEED)

## 2. Trainingsdaten

Wir ziehen zufällige Startwinkel $\varphi \in [0°, 360°)$. Für jeden Startwinkel entstehen:

- **Input:** $\sin(\varphi),\ \sin(\varphi + 30°),\ \dots,\ \sin(\varphi + 120°)$
- **Label:** $\sin(\varphi + 150°)$, also der Wert 30° nach dem letzten Input

In [ ]:
def make_dataset(n: int, n_inputs: int = N_INPUTS, step_deg: float = STEP_DEG,
                 noise_std: float = 0.0, rng: np.random.Generator = rng):
    """Erzeugt Trainingsdaten (Input-Fenster, Label) aus einer Sinusschwingung.

    Args:
        n: Anzahl der Beispiele.
        n_inputs: Anzahl der Samples pro Eingangsfenster.
        step_deg: Abstand zwischen zwei Samples in Grad.
        noise_std: Standardabweichung von Gauß-Rauschen auf den Inputs.
        rng: Zufallsgenerator.

    Returns:
        Tupel (X, y) mit X der Form (n, n_inputs) und y der Form (n,).
    """
    start = rng.uniform(0, 360, size=(n, 1))                     # zufällige Phase
    angles = start + step_deg * np.arange(n_inputs + 1)          # n_inputs Inputs + 1 Label
    values = np.sin(np.deg2rad(angles))
    X, y = values[:, :-1], values[:, -1]
    if noise_std > 0:
        X = X + rng.normal(0, noise_std, X.shape)
    return X, y


X_train, y_train = make_dataset(N_TRAIN, noise_std=NOISE_STD)
print("X_train:", X_train.shape, " y_train:", y_train.shape)
print("Erstes Beispiel:\n  Input:", np.round(X_train[0], 3), "\n  Label:", np.round(y_train[0], 3))

## 3. Neuronales Netz trainieren

Ein fully connected Netz (`MLPRegressor`) mit 5 Eingangs- und 1 Ausgangsneuron.

In [ ]:
model = MLPRegressor(hidden_layer_sizes=HIDDEN, activation="tanh", solver="adam",
                     max_iter=2000, tol=1e-8, n_iter_no_change=50, random_state=SEED)
model.fit(X_train, y_train)

print(f"Aufbau: {N_INPUTS} -> {' -> '.join(map(str, HIDDEN))} -> 1")
print(f"Trainingsiterationen: {model.n_iter_},  finaler Loss: {model.loss_:.2e}")

fig, ax = plt.subplots(figsize=(6, 3))
ax.semilogy(model.loss_curve_)
ax.set_xlabel("Iteration"); ax.set_ylabel("Loss (MSE/2)"); ax.set_title("Trainingsverlauf")
ax.grid(True)
plt.show()

## 4. Auto-regressive Prognose

Die Funktion `predict_autoregressive` ruft das Modell mehrfach nacheinander auf und füttert dabei
jeweils die letzte Prognose in das Eingangsfenster zurück.

In [ ]:
def predict_autoregressive(model, window: np.ndarray, n_steps: int) -> np.ndarray:
    """Prognostiziert n_steps Schritte in die Zukunft, indem das Modell auto-regressiv genutzt wird.

    Args:
        model: Trainiertes Modell mit `predict`-Methode (Eingang: (1, n_inputs)).
        window: Letzte Samples (Länge n_inputs), ältester zuerst.
        n_steps: Anzahl der Prognoseschritte (jeder Schritt = ein Abtastintervall).

    Returns:
        Array der Länge n_steps mit den Prognosen (Schritt +1, +2, ...).
    """
    window = np.asarray(window, dtype=float).copy()
    preds = []
    for _ in range(n_steps):
        y_next = float(model.predict(window.reshape(1, -1))[0])
        preds.append(y_next)
        window = np.append(window[1:], y_next)   # ältesten Wert raus, Prognose rein
    return np.array(preds)

## 5. Test an einer Beispiel-Sinusschwingung  ✏️

Wir erzeugen eine Sinuskurve mit einer **anderen Phase** als in der Trainingsmenge zufällig gezogen
wurde, legen den aktuellen Zeitpunkt $t$ fest und prognostizieren **+30°, +60° und +90°**.

In [ ]:
PHASE_DEG = 20      # Phasenverschiebung der Testschwingung
T_NOW_DEG = 360     # aktueller Zeitpunkt (Winkel) der Prognose
TEST_NOISE = 0.0    # Messrauschen auf den Input-Samples der Testschwingung

In [ ]:
sine = lambda deg: np.sin(np.deg2rad(deg + PHASE_DEG))

# Sample-Zeitpunkte: letzte 5 Samples im Abstand von 30° (das letzte liegt bei T_NOW_DEG)
sample_deg = T_NOW_DEG - STEP_DEG * np.arange(N_INPUTS - 1, -1, -1)
samples = sine(sample_deg) + rng.normal(0, TEST_NOISE, N_INPUTS)

# Prognose bei +30°, +60°, +90°
n_steps = 3
pred_deg = T_NOW_DEG + STEP_DEG * np.arange(1, n_steps + 1)
preds = predict_autoregressive(model, samples, n_steps)

print(f"{'Horizont':>9} | {'Prognose':>9} | {'Wahrer Wert':>11} | {'Fehler':>8}")
for h, p, d in zip(STEP_DEG * np.arange(1, n_steps + 1), preds, pred_deg):
    print(f"{'+' + str(h) + '°':>9} | {p:9.4f} | {sine(d):11.4f} | {abs(p - sine(d)):8.4f}")

In [ ]:
deg = np.linspace(0, 540, 1000)

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(deg, sine(deg), color="lightgray", linewidth=2, label="Sinusschwingung (wahr)")
ax.plot(sample_deg, samples, "o", color="tab:blue", markersize=8, label="Samples (Modell-Input)")
ax.plot(pred_deg, preds, "r*", markersize=15, label="Prognose (+30°, +60°, +90°)")
ax.plot(pred_deg, sine(pred_deg), "kx", markersize=9, label="wahre Werte")
for d, p, h in zip(pred_deg, preds, (30, 60, 90)):
    ax.annotate(f"+{h}°", (d, p), textcoords="offset points", xytext=(6, 8), color="red")
ax.axvline(T_NOW_DEG, color="gray", linestyle="--")
ax.text(T_NOW_DEG + 3, -1.25, "jetzt (t)", color="gray")
ax.set_xlabel("Winkel [°]"); ax.set_ylabel("Amplitude")
ax.set_ylim(-1.4, 1.4)
ax.set_xticks(range(0, 541, 30))
ax.grid(True); ax.legend(loc="upper left", ncol=2, fontsize=9)
ax.set_title("Statische Prognose mit auto-regressivem Modell")
plt.show()

## 6. Wie gut ist die Prognose bei größerem Horizont?

Wir testen über viele Zeitpunkte und Phasen den mittleren Fehler in Abhängigkeit vom Prognosehorizont.

In [ ]:
N_STEPS_MAX = 12
start = np.random.default_rng(1).uniform(0, 360, 500)    # 500 zufällige Phasen

errors = np.zeros((500, N_STEPS_MAX))
for i, s in enumerate(start):
    window = np.sin(np.deg2rad(s + STEP_DEG * np.arange(N_INPUTS)))
    p = predict_autoregressive(model, window, N_STEPS_MAX)
    truth = np.sin(np.deg2rad(s + STEP_DEG * np.arange(N_INPUTS, N_INPUTS + N_STEPS_MAX)))
    errors[i] = np.abs(p - truth)

horizons = STEP_DEG * np.arange(1, N_STEPS_MAX + 1)
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(horizons, errors.mean(axis=0), "o-")
ax.set_xlabel("Prognosehorizont [°]"); ax.set_ylabel("mittlerer absoluter Fehler")
ax.set_title("Fehler wächst mit dem Horizont"); ax.grid(True)
plt.show()

## Fragen zur Diskussion

- Warum wächst der Fehler mit dem Horizont? Was passiert mit dem Fehler einer Prognose, wenn sie als Input für die
  nächste dient (**Fehlerakkumulation**)?
- Setzen Sie `NOISE_STD` und `TEST_NOISE` auf z. B. `0.05`. Wie verändert sich die Prognose bei 90°?
- Trainieren Sie nur mit Startwinkeln aus $[0°, 90°)$. Wie gut ist die Prognose in anderen Bereichen?
  (Stichwort: **Generalisierung / Extrapolation**)
- Testen Sie eine Schwingung mit anderer Amplitude (z. B. $2\sin$). Was passiert? Warum?
- **Statisches vs. dynamisches Modell:** Ein rekurrentes Netz (RNN/LSTM) hat einen internen Zustand und
  braucht kein festes Eingangsfenster. Wo könnte das bei der Prognose von Fahrzeug- oder Ballbahnen helfen?